# Fase 0: fundamentos de redes neurais com NumPy
**Objetivo:** Compreender o funcionamento dos algoritmos fundamentais de redes neurais.

**Conceitos-chave:** regressão, gradiente descendente, neurônio, função de ativação, *backpropagation*.

### Sumário
*Etapa 0.1: Gradiente Descendente*

**Etapa 0.2: MLP do zero**

*Etapa 0.3: *Backpropagation**

*Etapa 0.4: Autograd simples*

*Mini-projeto 0: A curva de luz de uma supernova Ia*

In [1]:
import numpy as np

np.random.seed(42) # Garante que as células resultem nos mesmos valores se executadas em ordem

# Etapa 0.2 — Implementando uma MLP do zero

Uma rede neural (*multilayer perceptron* ou MLP) é composta por camadas sequenciais de neurônios cujo objetivo é, em geral, "aprender" um padrão e ser capaz de reproduzi-lo.

## 0.2.1 Camadas e neurônios
Para começar, precisamos entender o que é um **neurônio**. Ele é a menor unidade de uma rede neural, é nele que ocorre o processamento dos dados. Um neurônio recebe, de outro neurônio, uma entrada (podendo ela ser os próprios dados ou a saída de outro neurônio) e produz uma saída que será enviada como a entrada do próximo neurônio. Todo neurônio faz a mesma coisa: produz uma **cominação linear** da entrada. Seja $x$ um vetor de entrada e $y$ o vetor de saída, o que um neurônio faz é
$$
y_{i} = w x_{i}+b,
$$
em que $w$ representa o **peso** daquele neurônio e $b$ é seu bias. Matricialmente, podemos escrever que
$$
\mathbf{\hat{y}} = w\mathbf{x} + b.
$$
No fim, a tarefa de cada neurônio é produzir um número que corresponde a uma combinação linear dos valores de entrada. A partir de agora, nossas *arrays* sempre terão duas dimensões e nossos dados sempre serão **matrizes coluna**. Vamos implementar um neurônio simples.

In [2]:
x = np.array([[1], [2], [3], [4], [5]]) # Matriz dos dados com tamanho 5x1
w = np.random.randn(1, 1) # Inicializa uma matriz 1x1 (escalar) preenchida por valores sorteados de uma distribuição normal (mu = 0, simga = 1)
b = np.random.randn(1, 1) # Inicializa uma matriz de tamanho 1x1 sorteada da mesma distribuição normal

y = w * x + b

print(f'Dados:\n x =\n {x}\n')
print(f'Configurações do neurônio:\n w = {w}\n b = {b}\n')
print(f'Saída:\n y =\n{y}')

Dados:
 x =
 [[1]
 [2]
 [3]
 [4]
 [5]]

Configurações do neurônio:
 w = [[0.49671415]]
 b = [[-0.1382643]]

Saída:
 y =
[[0.35844985]
 [0.855164  ]
 [1.35187816]
 [1.84859231]
 [2.34530646]]


O caso que acabamos de implementar é uma rede neural com apenas uma camada escondida de um neurônio. As camadas escondidas são as camadas de neurônios que ficam entre o nuerônio de entrada e o neurônio de saída. Implementar mais camadas é tão simples quanto a anterior. A lógica é que os dados entram no primeiro neurônio $N_{1}$, passam pela transformação linear e são passados para o segundo neurônio, $N_{2}$, que executa a segunda transformação linear e nos devolve a saída. A lógica, então é
$$
\mathbf{x} \quad \longrightarrow \quad N_{1}:\quad w_{1}\mathbf{x} + b_{1}\quad \longrightarrow \quad \mathbf{h_{1}} \quad \longrightarrow \quad N_{2}:\quad w_{2}\mathbf{h_{1}} + b_{2} \quad \longrightarrow \quad \mathbf{\hat{y}}
$$

In [3]:
x = np.array([[1], [2], [3], [4], [5]])

# Configurações do neurônio 1
w1 = np.random.randn(1, 1)
b1 = np.random.randn(1, 1)

# Configurações do neurônio 2
w2 = np.random.randn(1, 1)
b2 = np.random.randn(1, 1)

# Enviando os dados para o neurônio 1
h1 = w1 * x + b1

# Enviando a saída do neurônio 1 para o neurônio 2
y = w2 * h1 + b2

print(f'Dados:\n x =\n {x}\n')
print(f'Configurações do neurônio 1:\n w1 = {w1}\n b1 = {b1}\n')
print(f'Saída do neurônio 1:\n h1 =\n{h1}\n')
print(f'Configurações do neurônio 2:\n w2 = {w2}\n b2 = {b2}\n')
print(f'Saída do neurônio 2:\n y =\n{y}\n')

Dados:
 x =
 [[1]
 [2]
 [3]
 [4]
 [5]]

Configurações do neurônio 1:
 w1 = [[0.64768854]]
 b1 = [[1.52302986]]

Saída do neurônio 1:
 h1 =
[[2.17071839]
 [2.81840693]
 [3.46609547]
 [4.11378401]
 [4.76147255]]

Configurações do neurônio 2:
 w2 = [[-0.23415337]]
 b2 = [[-0.23413696]]

Saída do neurônio 2:
 y =
[[-0.74241799]
 [-0.89407645]
 [-1.04573491]
 [-1.19739337]
 [-1.34905182]]



Assim, podemos criar quantas camadas quisermos, basta sequenciarmos as transformações dos neurônios.

## 0.2.2 *Forward propagation*

Essa é a primeira etapa do processamento de uma rede neural, e é o que acabamos de fazer. O *forward propagation* é o procedimento de "caminhar" pela rede neural no sentido da entrada para a saída, ou seja, passar os dados para frente. É isso o que fazemos quando queremos que a rede neural avalie o valor do modelo. Foi isso que fizemos no caso acima, passamos os dados de entrada para o primeiro neurônio, depois passamos para o segundo neurônio e obtivemos nossa saída.

A ideia, agora, é ver isso com mais detalhes. Vamos criar uma rede neural com duas camadas, mas cada uma contendo $4$ neurônios. Nosso formalismo agora será totalmente matricial. Vamos analisar, passo a passo, como nosso *forward propagation* funcionará. Como cada neurônio possui seu próprio peso, e teremos múltiplos neurônios em cada camada, iremos associar, a cada camada, uma matrix $\mathbf{W}$ que armazenará os pesos dos neurônios daquela camada. A dimensão dessas matriz são *muito importantes* e estão relacionadas a quantos neurônios informarão a entrada a essa camada e para quantos neurônios essa camada enviará as saídas. Vamos exemplificar com uma rede de duas camadas. Uma camada será uma **camada escondida**, aquela em que "não vemos" o que acontece, e uma camada de saída, que nos fornecerá o resultado.

Temos nossa matriz de dados $\mathbf{x}$ cuja dimensão é $5\times 1$, isso significa que eu possuo $5$ amostras de $1$ tipo de dado (o valor de $\mathbf{x}$). Essa matriz deverá ser passada para *cada* neurônio da primeira camada e cada neurônio deverá aplicar sua transformação linear em cada amostra dos dados. Sendo assim, a matriz de pesos $\mathbf{W_{1}}$ deve ter dimensão $1\times 4$. Isso nos diz que essa camada receberá valores de $1$ neurônio e produzirá $4$ saídas, uma por neurônio, ou seja, temos quatro conexões (cada uma ligando o neurônio de entrada a um neurônio da camada). A saída será salva em uma matriz $\mathbf{H}_{1}$ cuja dimensão é $5\times 4$, com cada linha representando uma das amostras e, cada coluna, um dos neurônios. Em seguida, a matriz $H_{1}$ passará para o neurônio da segunda camada, a camada de saída, representado pela matriz $W_{2}$. A segunda camada receberá informação de $4$ neurônios e produzirá $1$ saída, ou seja, temos, de novo, quatro conexões e, portanto, quatro pesos. Assim, a dimensão de $W_{2}$ é $4\times 1$. Ao final, teremos uma matriz $\mathbf{y}$ cuja dimensão é $5\times 1$, igual ao neurônio de entrada. O processo, então, é
$$
\mathbf{x} \quad \longrightarrow \quad N_{1}: \quad \mathbf{x}\mathbf{W_{1}} + \mathbf{b}_{1} \quad \longrightarrow \quad \mathbf{H}_{1} \quad \longrightarrow \quad N_{2}:\quad \mathbf{H}_{1}\mathbf{W}_{2}+\mathbf{b}_{2} \quad \longrightarrow \quad \mathbf{\hat{y}},
$$
em que as multiplicações são matriciais agora. Quanto aos *bias*, $\mathbf{b}_{1}$ possui dimensão $1\times 4$, com cada coluna representando o *bias* de um neurônio. Esse vetor linha será somado igualmente a todas as linhas da matriz $\mathbf{x}\mathbf{W_{1}}$. Já $\mathbf{b}_{2}$ possui dimensão $1\times 1$, pois há apenas um neurônio. Assim como no caso da primeira camada, esse elemento será somado a todas as linhas da matriz $\mathbf{H_{1}}\mathbf{W_{2}}$.

Vamos fazer essa implementação via classes. Primeiro, vamos criar uma classe `Camada`.

In [4]:
class Camada:
    ''' Define uma camada de neurônios.
    
    Atributos:
        n_entrada: número de entradas recebidas por cada neurônio da camada.
        n_saida: número de neurônios desta camada.        
    '''
    # Inicialização de uma camada
    def __init__(self, n_entrada, n_saida):
        self.n_entrada = n_entrada
        self.n_saida = n_saida

        self.pesos = np.random.randn(n_entrada, n_saida) # Gera a matriz de pesos cuja dimensão é n_entrada x n_saida
        self.bias = np.random.randn(1, n_saida) # Gera a matriz de bias cuja dimensão é 1 x n_saida

    # Vamos definir um método `forward` para executar o forward propagation
    def forward(self, entrada):
        ''' Executa o foward propagation pela camada.
        Parâmetros:
            entrada: valores que serão processados pelos neurônios da camada.
        '''
        M1 = np.dot(entrada, self.pesos) # Realiza a multiplicação da matriz de entrada pela matriz de pesos da camada
        M2 = M1 + self.bias # Soma a matriz de bias a todas as linhas de M1
        return M2        

Agora, vamos gerar as nossas camadas e fazer o *forward propagation* manualmente.

In [5]:
camada1 = Camada(1, 4) # Inicializa uma camada que receberá valores de 1 neurônio e produzirá 4 saídas (ou seja, possui 4 neurônios)
camada_saida = Camada(4, 1) # Inicializa uma camada que receberá valores de 4 neurônios e produzirá 1 saída (ou seja, possui 1 neurônio)

print(f'Pesos e bias iniciais da camada 1:\nW1 = {camada1.pesos}\nb1 = {camada1.bias}\n')
print(f'Pesos iniciais da camada de saída:\nW2 =\n{camada_saida.pesos}\nb2 = {camada_saida.bias}')

Pesos e bias iniciais da camada 1:
W1 = [[ 1.57921282  0.76743473 -0.46947439  0.54256004]]
b1 = [[-0.46341769 -0.46572975  0.24196227 -1.91328024]]

Pesos iniciais da camada de saída:
W2 =
[[-1.72491783]
 [-0.56228753]
 [-1.01283112]
 [ 0.31424733]]
b2 = [[-0.90802408]]


In [6]:
x = np.array([[1], [2], [3], [4], [5]])

# Fazendo o foward propagation pela camada 1
H1 = camada1.forward(x)

print(f'O forward propagation pela camada 1 resultou em\n H1 =\n{H1}')

# Fazendo o foward propagation pela camada de saída
y = camada_saida.forward(H1)
print(f'\nO forward propagation pela camada de saída resultou em\n y =\n{y}')

O forward propagation pela camada 1 resultou em
 H1 =
[[ 1.11579512  0.30170498 -0.22751211 -1.3707202 ]
 [ 2.69500794  1.0691397  -0.6969865  -0.82816016]
 [ 4.27422075  1.83657443 -1.16646089 -0.28560011]
 [ 5.85343357  2.60400916 -1.63593527  0.25695993]
 [ 7.43264638  3.37144389 -2.10540966  0.79951997]]

O forward propagation pela camada de saída resultou em
 y =
[[ -3.20263774]
 [ -5.71217275]
 [ -8.22170776]
 [-10.73124277]
 [-13.24077778]]


## 0.2.3 A função de ativação

Sabemos como percorrer uma rede neural no sentido da entrada para a saída, ou seja, sabemos como gerar um resultado a partir de uma entrada usando camadas de neurônios. Resumidamente, realizamos várias transformações lineares em sequência. O problema é que transformações lineares em sequência nada mais são do que uma grande transformação linear. Por exemplo, considere o caso simples de duas camadas e um neurônio em cada. A entrada $x$ é fornececida para a primeira cada, resultando em
$$
\mathbf{H}_{1} = \mathbf{x}w_{1} + b_{1}.
$$
Em seguida, passaremos $\mathbf{H}_{1}$ para o neurônio de saída, resultando em
$$
\mathbf{\hat{y}} = \mathbf{H}_{1}w_{2} + b_{2}.
$$
Substituindo $\mathbf{H}_{1}$, obtemos que
$$
\mathbf{\hat{y}} = \mathbf{x}w_{1}w_{2} + b_{1}w_{2} + b_{2}.
$$
Ou seja, $\mathbf{\hat{y}}$ nada mais é do que uma tranformação linear de $\mathbf{x}$. Isso é um problema porque, assim, só seremos capazes de aproximar funções lineares, mas queremos aproximar vários tipos de função. Por isso, é necessário que, de algum jeito, criemos alguma não-linearidade na rede neural. Isso é feito por meio da **função de ativação**.

Basicamente, a função de ativação é uma função matemática não-linear que é a responsável por justamente criar a não-linearidade no modelo. Existem várias funções de ativação, as mais comuns sendo as funções ReLU, sigmoide e tangente hiperbólica. Uma das mais utilizadas é a função ReLU, devido ao formato da função: enquanto as outras duas atingem "platôs" para valores altos, reduzindo drasticamente o valor do gradiente da função, a função ReLU não tem esse problema. A função ReLU é definida como
$$
z = \max (0, x).
$$
É uma função bem simples de implementar, tornando a inserção de não-linearidade no modelo praticamente trivial. Geralmente, a função de ativação é inserida *após* as camadas escondidas. Para o nosso exemplo de duas camadas, o fluxo agora é
$$
\mathbf{x} \quad \longrightarrow \quad N_{1}: \quad \mathbf{x}\mathbf{W}_{1} + \mathbf{b}_{1} \quad \longrightarrow \quad \mathbf{H}_{1} \quad \longrightarrow \quad \mathbf{z} = \max(0, \mathbf{H}_{1}) \quad \longrightarrow \quad N_{2}: \quad \mathbf{z}\mathbf{W}_{2} + \mathbf{b}_{2} \quad \longrightarrow \quad \mathbf{\hat{y}},
$$
em que $\mathbf{z}$ é uma matriz de dimensão $5\times 4$ que contém o resultado da função aplicada **elemento a elemento** da matriz $\mathbf{H}_{1}$.

Vamos implementar uma rede neural com função de ativação ReLU.

In [7]:
np.random.seed(42) # Fixando a seed para comparar esse resultado com a implementação do MLP nas células seguintes

camada1 = Camada(1, 4)
camada_saida = Camada(4, 1)

x = np.array([[1], [2], [3], [4], [5]])

# Fazendo o foward propagation pela camada 1
H1 = camada1.forward(x)

print(f'O forward propagation pela camada 1 resultou em\n H1 =\n{H1}\n')

zeros = np.zeros_like(H1) # Gera uma matriz com dimensões de H1 preenchida com zeros
z = np.maximum(zeros, H1) # Seleciona, elemento a elemento, aqueles que são maiores que zero

print(f'O resultado da função de ativação da camada 1 é\n z =\n{z}\n')

# Fazendo o foward propagation pela camada de saída
y = camada_saida.forward(z)
print(f'O forward propagation pela camada de saída resultou em\n y =\n{y}')

O forward propagation pela camada 1 resultou em
 H1 =
[[ 0.26256078 -0.37240126  2.22690135  2.29046459]
 [ 0.75927493 -0.51066556  2.87458989  3.81349444]
 [ 1.25598908 -0.64892986  3.52227843  5.3365243 ]
 [ 1.75270324 -0.78719416  4.16996697  6.85955415]
 [ 2.24941739 -0.92545846  4.81765551  8.38258401]]

O resultado da função de ativação da camada 1 é
 z =
[[0.26256078 0.         2.22690135 2.29046459]
 [0.75927493 0.         2.87458989 3.81349444]
 [1.25598908 0.         3.52227843 5.3365243 ]
 [1.75270324 0.         4.16996697 6.85955415]
 [2.24941739 0.         4.81765551 8.38258401]]

O forward propagation pela camada de saída resultou em
 y =
[[-1.98002628]
 [-3.2226915 ]
 [-4.46535672]
 [-5.70802194]
 [-6.95068716]]


## 0.2.4 Implementando um *multilayer perceptron*

Em essência, já implementamos um MLP acima. O que faremos, agora, é formalizar essa implementação para que ela possa ser executada em *loop*, e não manualmente igual fizemos. Para isso, criaremos uma nova classe chamada `MLP` que terá, especificamente, duas camadas.

In [8]:
class MLP:
    ''' Define um multilayer perceptron de duas camadas.
    Atributos:
        camada1: a primeira camada do MLP.
        camada_saida: a camada de saída do MLP.

    Parâmetros:
        dim_n1: lista ou tupla contendo as dimensões da primeira camada na forma (n_entrada, n_saida).
        dim_n1: lista ou tupla contendo as dimensões da camada de saída na forma (n_entrada, n_saida).
    '''

    def __init__(self, dim_n1, dim_n2):
        self.camada1 = Camada(dim_n1[0], dim_n1[1])
        self.camada_saida = Camada(dim_n2[0], dim_n2[1])

    # Definindo a função ReLU para ser usada como função de ativação
    def relu(self, entrada):
        zeros = np.zeros_like(entrada)
        return np.maximum(zeros, entrada)

    # Definindo o método para executar o forward propagation pela rede
    def forward(self, entrada):
        H1 = self.camada1.forward(entrada) # Passa para a camada 1
        z = self.relu(H1) # Passa a saída da camada 1 pela função de ativação
        y = self.camada_saida.forward(z) # Passa a saída da função de ativação para a camada de saída
        return y

Vamos testar, mais uma vez, o mesmo exemplo.

In [9]:
np.random.seed(42)

x = np.array([[1], [2], [3], [4], [5]])

dim_n1 = (1, 4) # Dimensão da camada 1
dim_n2 = (4, 1) # Dimensão da camada de saída

# Definindo nosso MLP
mlp = MLP(dim_n1, dim_n2)

# Executando o forward propagation pela rede neural
y = mlp.forward(x)

print(f'A saída da rede neural é y =\n{y}')

A saída da rede neural é y =
[[-1.98002628]
 [-3.2226915 ]
 [-4.46535672]
 [-5.70802194]
 [-6.95068716]]
